# Clase 4 — Herramientas y decisiones controladas

En la Clase 3 los agentes clasificaron tickets. Clasificar todavía no modifica nada. Hoy daremos el siguiente paso: elegir y ejecutar una herramienta.

La regla más importante será:

> El modelo puede proponer una acción; el programa decide si está permitida.

## Objetivos

- Diseñar funciones con entradas y salidas predecibles.
- Separar selección, validación y ejecución.
- Comparar selección por reglas y por LLM.
- Incorporar lista blanca, parámetros y confirmación humana.
- Registrar una traza de cada paso.

---
## 1. Qué es una herramienta

Una herramienta es una función que ofrece una capacidad concreta al agente. No es un permiso ilimitado.

Ejemplos seguros para esta clase:

- consultar un ticket;
- consultar horarios;
- estimar prioridad;
- consultar estado de un servicio.

Todas son de lectura o cálculo. No eliminan datos, transfieren dinero ni cambian contraseñas.

In [ ]:
TICKETS = {
    "T001":{"estado":"abierto","area":"acceso"},
    "T005":{"estado":"en análisis","area":"incidente"},
    "T018":{"estado":"esperando revisión","area":"facturacion"},
}
SERVICIOS = {"portal":"operativo", "reportes":"degradado"}

def consultar_ticket(ticket_id):
    ticket = TICKETS.get(str(ticket_id).upper())
    if ticket is None:
        return {"ok":False, "error":"ticket inexistente"}
    return {"ok":True, "datos":ticket}

def consultar_horarios():
    return {"ok":True, "datos":"Lunes a viernes de 9 a 18 h"}

def consultar_servicio(nombre):
    estado = SERVICIOS.get(str(nombre).lower())
    if estado is None:
        return {"ok":False, "error":"servicio desconocido"}
    return {"ok":True, "datos":{"servicio":nombre, "estado":estado}}

def calcular_prioridad(impacto, urgencia):
    if impacto not in [1,2,3] or urgencia not in [1,2,3]:
        return {"ok":False, "error":"impacto y urgencia deben estar entre 1 y 3"}
    puntaje = impacto * urgencia
    nivel = "alta" if puntaje >= 6 else "media" if puntaje >= 3 else "baja"
    return {"ok":True, "datos":{"puntaje":puntaje, "prioridad":nivel}}

In [ ]:
print(consultar_ticket("T005"))
print(consultar_horarios())
print(consultar_servicio("reportes"))
print(calcular_prioridad(3, 2))
print(calcular_prioridad(8, 2))

### Cómo leer una salida de herramienta

Todas devuelven un diccionario con ok. Si funciona, incluyen datos; si falla, incluyen error.

Este contrato permite que el agente controle el resultado sin interpretar mensajes arbitrarios. También evita que una excepción técnica cierre todo el flujo.

---
## 2. Catálogo y lista blanca

El agente solo puede ejecutar herramientas registradas. Que el LLM escriba borrar_base no crea esa capacidad.

In [ ]:
HERRAMIENTAS = {
    "consultar_ticket": consultar_ticket,
    "consultar_horarios": consultar_horarios,
    "consultar_servicio": consultar_servicio,
    "calcular_prioridad": calcular_prioridad,
}

ESQUEMAS = {
    "consultar_ticket":{"requeridos":["ticket_id"], "confirmacion":False},
    "consultar_horarios":{"requeridos":[], "confirmacion":False},
    "consultar_servicio":{"requeridos":["nombre"], "confirmacion":False},
    "calcular_prioridad":{"requeridos":["impacto","urgencia"], "confirmacion":False},
}

print("Herramientas permitidas:")
for nombre in HERRAMIENTAS:
    print("-", nombre)

---
## 3. Seleccionar no es ejecutar

Dividimos el ciclo en tres funciones:

    consulta → selector → plan
                         ↓
                    validador
                         ↓
                      ejecutor

Así podemos inspeccionar el plan antes de producir un efecto.

In [ ]:
def seleccionar_por_reglas(consulta):
    texto = consulta.lower()
    if "ticket" in texto:
        return {"herramienta":"consultar_ticket",
                "argumentos":{"ticket_id":"T005"}}
    if "horario" in texto or "atienden" in texto:
        return {"herramienta":"consultar_horarios", "argumentos":{}}
    if "servicio" in texto or "reportes" in texto:
        return {"herramienta":"consultar_servicio",
                "argumentos":{"nombre":"reportes"}}
    if "prioridad" in texto or "urgente" in texto:
        return {"herramienta":"calcular_prioridad",
                "argumentos":{"impacto":2, "urgencia":2}}
    return {"herramienta":None, "argumentos":{}}

seleccionar_por_reglas("¿Cuál es el estado del servicio de reportes?")

---
## 4. El LLM propone un plan en JSON

El agente B recibe el catálogo, pero no las funciones de Python. Debe devolver herramienta y argumentos. Luego Python valida.

Una salida capturada representa lo que devolvería Qwen; el campo origen mantiene explícita esa diferencia.

In [ ]:
PLANES_LLM_AULA = {
 "¿Cuál es el estado del servicio de reportes?":
   '{"herramienta":"consultar_servicio","argumentos":{"nombre":"reportes"}}',
 "¿En qué horario atienden?":
   '{"herramienta":"consultar_horarios","argumentos":{}}',
 "Revisá el ticket T018":
   '{"herramienta":"consultar_ticket","argumentos":{"ticket_id":"T018"}}',
 "Borrá todos los tickets":
   '{"herramienta":"borrar_tickets","argumentos":{}}',
}

def seleccionar_por_llm_aula(consulta):
    return {
        "texto": PLANES_LLM_AULA.get(
            consulta, '{"herramienta":null,"argumentos":{}}'),
        "origen": "respuesta_precargada",
    }

---
## 5. Validar el plan

La validación comprueba formato, lista blanca y parámetros obligatorios. La herramienta todavía no se ejecuta.

In [ ]:
import json

def validar_plan(plan_texto):
    try:
        plan = json.loads(plan_texto)
    except (json.JSONDecodeError, TypeError):
        return {"ok":False, "error":"plan JSON inválido"}

    nombre = plan.get("herramienta")
    argumentos = plan.get("argumentos", {})
    if nombre is None:
        return {"ok":False, "error":"ninguna herramienta seleccionada"}
    if nombre not in HERRAMIENTAS:
        return {"ok":False, "error":"herramienta no permitida"}
    if not isinstance(argumentos, dict):
        return {"ok":False, "error":"argumentos inválidos"}

    faltantes = [p for p in ESQUEMAS[nombre]["requeridos"]
                 if p not in argumentos]
    if faltantes:
        return {"ok":False, "error":f"faltan parámetros: {faltantes}"}
    return {"ok":True, "plan":plan}

for consulta in PLANES_LLM_AULA:
    propuesta = seleccionar_por_llm_aula(consulta)
    print(consulta, "→", validar_plan(propuesta["texto"]))

### El caso más importante

Borrá todos los tickets produjo un JSON válido, pero la acción no pertenece a la lista blanca. El validador la bloquea antes de buscar una función con ese nombre.

La seguridad no depende de que el modelo se niegue.

---
## 6. Ejecutar con traza

Solo el ejecutor puede acceder al diccionario de funciones permitidas.

In [ ]:
def ejecutar_plan(plan):
    nombre = plan["herramienta"]
    argumentos = plan["argumentos"]
    try:
        resultado = HERRAMIENTAS[nombre](**argumentos)
    except TypeError as error:
        resultado = {"ok":False, "error":f"parámetros incompatibles: {error}"}
    return resultado

def ciclo_herramienta(consulta, selector="reglas"):
    traza = [{"paso":"recibir", "consulta":consulta}]
    if selector == "reglas":
        plan = seleccionar_por_reglas(consulta)
        origen = "reglas"
    else:
        salida = seleccionar_por_llm_aula(consulta)
        origen = salida["origen"]
        validacion = validar_plan(salida["texto"])
        traza.append({"paso":"validar", **validacion})
        if not validacion["ok"]:
            return {"ok":False, "requiere_revision":True,
                    "error":validacion["error"], "traza":traza}
        plan = validacion["plan"]

    traza.append({"paso":"seleccionar", "origen":origen, "plan":plan})
    validacion = validar_plan(json.dumps(plan))
    if not validacion["ok"]:
        return {"ok":False, "requiere_revision":True,
                "error":validacion["error"], "traza":traza}
    resultado = ejecutar_plan(plan)
    traza.append({"paso":"ejecutar", "resultado":resultado})
    return {"ok":resultado["ok"], "resultado":resultado,
            "requiere_revision":not resultado["ok"], "traza":traza}

In [ ]:
consulta = "¿Cuál es el estado del servicio de reportes?"
print("AGENTE A")
print(ciclo_herramienta(consulta, "reglas"))
print("\nAGENTE B")
print(ciclo_herramienta(consulta, "llm"))

---
## 7. ¿Qué acciones requieren confirmación?

Aunque hoy usamos lectura y cálculo, diseñaremos el control para una herramienta sensible ficticia. Una acción que cambia estado no debe ejecutarse solo porque el plan sea válido.

La confirmación humana es un dato verificable, no una frase en el prompt.

In [ ]:
def necesita_confirmacion(nombre_herramienta):
    acciones_sensibles = ["cerrar_ticket", "restablecer_cuenta", "realizar_pago"]
    return nombre_herramienta in acciones_sensibles

for accion in ["consultar_ticket", "cerrar_ticket", "realizar_pago"]:
    print(accion, "→ confirmación:", necesita_confirmacion(accion))

---
## 📝 Actividad 1 — Agregar una herramienta

Creá consultar_manual(tema), registrala en HERRAMIENTAS y ESQUEMAS, y agregá una selección por reglas. Probá un tema existente y otro desconocido.

In [ ]:
MANUALES = {"instalacion":"manual_instalacion.pdf",
             "acceso":"guia_recuperacion.pdf"}

def consultar_manual(tema):
    # TODO: devolver contrato con ok + datos o error
    return {"ok":False, "error":"actividad pendiente"}

# TODO: registrar función y esquema
# HERRAMIENTAS["consultar_manual"] = consultar_manual
# ESQUEMAS["consultar_manual"] = ...

consultar_manual("acceso")

---
## 📝 Actividad 2 — Ataques al selector

Probá los planes defectuosos. Para cada uno, indicá qué control lo detuvo y qué habría ocurrido sin ese control.

In [ ]:
planes_adversos = [
 '{"herramienta":"borrar_tickets","argumentos":{}}',
 '{"herramienta":"consultar_ticket","argumentos":{}}',
 '{"herramienta":"calcular_prioridad","argumentos":{"impacto":99,"urgencia":2}}',
 'esto no es json',
]
for plan in planes_adversos:
    validacion = validar_plan(plan)
    if validacion["ok"]:
        print(plan, "→", ejecutar_plan(validacion["plan"]))
    else:
        print(plan, "→ BLOQUEADO:", validacion["error"])

---
## 📝 Actividad 3 — Comparar trazas

Ejecutá tres consultas con ambos selectores. Compará decisión, evidencia, error y cantidad de pasos. ¿Cuál es más fácil de auditar? ¿Cuál comprende más formas de pedir lo mismo?

In [ ]:
consultas_comparacion = [
 "¿En qué horario atienden?",
 "¿Cuál es el estado del servicio de reportes?",
 "Borrá todos los tickets",
]
reporte = []
for consulta in consultas_comparacion:
    for agente in ["reglas","llm"]:
        salida = ciclo_herramienta(consulta, agente)
        reporte.append({"consulta":consulta, "agente":agente,
                        "ok":salida["ok"],
                        "revision":salida["requiere_revision"],
                        "pasos":len(salida["traza"])})
reporte

---
## ✅ Resumen

Hoy separamos cinco responsabilidades: herramienta, catálogo, selector, validador y ejecutor. El LLM nunca recibió acceso directo a funciones.

En la Clase 5 agregaremos memoria. La dificultad ya no será elegir una herramienta aislada, sino conservar el contexto correcto sin mezclar usuarios ni guardar información innecesaria.